In [2]:
# Step 1b (diagnostic only): variance explained by a degree-3 polynomial of sample order. The corrected matrix is NOT used downstream.
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from google.colab import files


def load_and_preprocess_data(filepath):
    """Load pre-normalized DNA data from CSV file."""
    try:
        # Read the CSV file
        dna_df = pd.read_table(filepath, header=None)

        # Split the comma-separated values and convert to float
        if isinstance(dna_df[0].iloc[0], str):
            dna_df = pd.DataFrame([
                list(map(float, row.split(',')))
                for row in dna_df[0]
            ])

        return dna_df

    except Exception as e:
        print(f"Error loading data: {str(e)}")
        raise


def create_covariates(n_samples, degree=3):
    """
    Create polynomial covariates of sample order (row position) for a
    diagnostic check of technical trends along the sample order.
    """
    # Create sample index
    sample_idx = np.arange(n_samples)

    # Create polynomial features
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    covar = poly.fit_transform(sample_idx.reshape(-1, 1))

    return covar


def correct_confounding(DNA, covariates):
    """
    Fit a polynomial of sample order to each CpG and return the residuals.
    Also returns, per CpG, the fraction of variance explained by the polynomial (R2).
    Diagnostic only: the corrected matrix is NOT used downstream.
    """
    lm = LinearRegression(fit_intercept=True)

    corrected_data = np.zeros_like(DNA)
    r2 = np.zeros(DNA.shape[1])
    for j in range(DNA.shape[1]):
        lm.fit(covariates, DNA[:, j])
        corrected_data[:, j] = DNA[:, j] - lm.predict(covariates)
        r2[j] = lm.score(covariates, DNA[:, j])   # fraction of variance explained

        if j % 1000 == 0:
            print(f"Processed {j}/{DNA.shape[1]} CpGs")

    return corrected_data, r2


def main():
    # Upload input file
    print("Please upload your methylation CSV file (DNAm_processed_imputed.csv)...")
    uploaded = files.upload()
    input_file = list(uploaded.keys())[0]

    # Load data
    print("Loading DNA data...")
    DNA = load_and_preprocess_data(input_file)

    # Create covariates matrix based on number of patients
    print("Creating covariates...")
    covariates = create_covariates(DNA.shape[0])

    print("Starting confounding correction (diagnostic)...")
    try:
        corrected_data, r2 = correct_confounding(DNA.to_numpy(), covariates)

        # Report how much variance the polynomial of sample order explains
        print("\nVariance explained by the degree-3 polynomial of sample order (per CpG):")
        print(f"  mean   = {100 * r2.mean():.2f}%")
        print(f"  median = {100 * np.median(r2):.2f}%")
        print(f"  max    = {100 * r2.max():.2f}%")
        print("  The corrected matrix is not used in later steps.")

        # Convert to DataFrame and save
        output_file = "DNA_corrected.csv"
        res_df = pd.DataFrame(
            corrected_data,
            index=DNA.index,
            columns=DNA.columns
        )
        res_df.to_csv(output_file, sep=',', index_label='TargetID')

        files.download(output_file)

        print("\nResults preview:")
        print(res_df.iloc[:5, :5])

    except Exception as e:
        print(f"Error during processing: {str(e)}")
        raise


if __name__ == '__main__':
    main()

Please upload your methylation CSV file (DNAm_processed_imputed.csv)...


Saving DNAm_processed_imputed.csv to DNAm_processed_imputed.csv
Loading DNA data...
Creating covariates...
Starting confounding correction (diagnostic)...
Processed 0/2000 CpGs
Processed 1000/2000 CpGs

Variance explained by the degree-3 polynomial of sample order (per CpG):
  mean   = 0.14%
  median = 0.12%
  max    = 0.76%
  The corrected matrix is not used in later steps.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Results preview:
          0         1         2         3         4
0 -0.201711 -0.417403 -0.324783  0.515904 -0.301286
1 -0.194726 -0.035529  0.391862 -0.367105  0.376085
2 -0.200776 -0.382035 -0.253028  0.446115 -0.297410
3 -0.212346  0.534054  0.530461 -0.375870 -0.301921
4  0.560238  0.233051  0.230008  0.149811  0.433887
